# LLM setup



In [1]:
!pip uninstall -y datasets


Found existing installation: datasets 2.14.4
Uninstalling datasets-2.14.4:
  Successfully uninstalled datasets-2.14.4


In [1]:
import subprocess
import sys
from importlib.metadata import PackageNotFoundError, version

IN_COLAB = "google.colab" in sys.modules


def installed_version(package):
    try:
        return version(package)
    except PackageNotFoundError:
        return None

# Skipped once the packages are installed, so this cell can be re-run after restarting the session.
if IN_COLAB and installed_version("pandasai") != "3.0.0":
    if sys.version_info[:2] != (3, 11):
        raise RuntimeError("Choose a Python 3.11 Colab runtime before installing these packages.")
    # Install into the Python running these cells.
    subprocess.check_call([sys.executable, "-m", "pip", "install", "uv>=0.8,<1"])
    # Same versions as uv.lock (torch is preinstalled on Colab).
    packages = [
        "ipython>=8,<9",
        "matplotlib-inline<0.2",
        "numpy==1.26.4",
        "pandas==2.3.3",
        "transformers==5.17.0",
        "accelerate==1.15.0",
        "bitsandbytes==0.50.2",
        "peft==0.21.2",
        "sentence-transformers==6.1.0",
        "requests",
        "pandasai==3.0.0",
    ]
    subprocess.check_call([sys.executable, "-m", "uv", "pip", "install",
                           "--python", sys.executable, *packages])
    print("Installation finished. Choose Runtime → Restart session before continuing.")

In [2]:
%pip install faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 103.3 MB/s eta 0:00:00


## Get the project code from github

On Colab, clone your team's repository so that the `ift6758` package can be imported.
For a private repository, create a GitHub [personal access token](https://docs.github.com/en/authentication/keeping-your-account-and-data-secure/managing-your-personal-access-tokens) with read access and add it to Colab's *Secrets* (key icon in the left sidebar) as `GITHUB_TOKEN`. Never paste the token in the notebook.

In [24]:
import os
import subprocess
IN_COLAB=True
if IN_COLAB:
    from google.colab import userdata

    REPO = "aliciavaq/data_science_project"
    REPO_DIR = "/content/repo"
    if not os.path.exists(REPO_DIR):
        token = userdata.get("GITHUB_TOKEN")
        subprocess.check_call(["git", "clone", f"https://{token}@github.com/{REPO}.git", REPO_DIR])
    sys.path.insert(0, REPO_DIR)

In [26]:
!git -C /content/repo pull

Already up to date.


## Check the GPU

In [4]:
import torch

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("No CUDA GPU found. On Colab, select a T4 GPU runtime (Runtime → Change runtime type).")

GPU: Tesla T4


## Load the model

The first run downloads ~8 GB of weights to the Hugging Face cache.

In [5]:
from ift6758.llm import load_model

model, tokenizer = load_model()
print(f"Loaded on {model.device} ({model.dtype})")

config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.38k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/238 [00:00<?, ?B/s]

Loaded on cuda:0 (torch.float16)


In [6]:
#test if model interface works
messages = [{"role": "user", "content": "In one sentence, what is a hat trick in hockey?"}]
inputs = tokenizer.apply_chat_template(
    messages, add_generation_prompt=True, return_tensors="pt", return_dict=True
).to(model.device)
output_ids = model.generate(**inputs, max_new_tokens=64)
print(tokenizer.decode(output_ids[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True))

In hockey, a hat trick occurs when a player scores a goal, assists on two other goals, or achieves a combination of these three milestones in a single game.


#Code Generation
Using the retrieved documentation, prompt the provided Qwen 4B model to write a Python class or function that fetches, caches locally, and loads play-by-play data for a given season.

##RAG from NHL docs implementation


In [7]:
from ift6758.llm.rag import LLM_DIR, embedder,load_docs, build_index, retrieve, save_retrieval

docs = load_docs()
chunks, index = build_index(docs)
queries = [
    "endpoint to fetch play-by-play information for a specific game",
    "get the game schedule for a specific date",
]
context = "\n\n".join(retrieve(q, chunks, index, k=2) for q in queries)
save_retrieval(LLM_DIR / "retrieved.md", "\n".join(queries), context)
print(context)



modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

[1] ### Game Events
#### Get Play By Play
- **Endpoint**: `/v1/gamecenter/{game-id}/play-by-play`
- **Method**: GET
- **Description**: Retrieve play-by-play information for a specific game.
- **Parameters**:
  - `game-id` (int) - Game ID
- **Response**: JSON format

###### Example using cURL:

```bash
curl -X GET "https://api-web.nhle.com/v1/gamecenter/2023020204/play-by-play"
```

#### Get Landing
- **Endpoint**: `/v1/gamecenter/{game-id}/landing`
- **Method**: GET
- **Description**: Retrieve landing information for a specific game.
- **Parameters**:
  - `game-id` (int) - Game ID
- **Response**: JSON format

###### Example using cURL:

```bash
curl -X GET "https://api-web.nhle.com/v1/gamecenter/2023020204/landing"
```

[2] ### Additional Game Content

#### Get Game Right Rail Content
- **Endpoint**: `/v1/gamecenter/{game-id}/right-rail`
- **Method**: GET
- **Description**: Retrieves sidebar content for the game center view.
- **Parameters**:
  - `game-id` (int) - Game ID
- **Response*

In [23]:
from ift6758.llm.qwen import ask_qwen
from IPython.display import  Markdown
game_id_note="""
Game_id will look like this: 2023020001
The first 4 digits identify the season ('year')  of the game (ie. 2017 for the 2017-2018 season). Always refer to a season with the starting year. A game played in March 2018 would still have a game ID that starts with 2017
The next 2 digits give the type of game .For a season those are : 02 = regular season, 03 = playoffs
The final 4 digits identify the specific game number.

"""
code_prompt = f"""NHL API documentation:
    {context}

NOTE: {game_id_note}

OBSERVED API RESPONSES:
The API host is https://api-web.nhle.com.
- GET /v1/schedule/YYYY-MM-DD returns the 7 days starting at that date. 'gameWeek' is a list of DAYS, each with a
  'games' list; each game has 'id' (the full game_id int) and 'gameType' (an int).
- GET /v1/gamecenter/{{game_id}}/play-by-play returns {{"id": <same int game id>, "plays": [...]}}.


TASK: Write a function `get_hockey_data(year: int, path: str = None) -> pandas.DataFrame` that fetches, caches
locally, and loads the play-by-play data of all regular-season and playoff games of the season starting in `year`.
Follow all of the specifications below. Use a helper if necessary.

SPECIFICATIONS:
1. Find the games by querying the schedule every 7 days from September 1 of `year` to October 31 of `year + 1`,
   looping over the 7 days of every gameWeek. Keep games with gameType 2 or 3 whose id starts with str(year)
   ids must be ints exactly as returned. Collect ids in a set. NEVER build or guess ids.
2. If path is None, use the HOCKEY_DATA_DIR environment variable. Raise ValueError if neither is set.
3. Cache each game as <path>/<season>/<game_id>.json (a season could be "2022-23"). If the file already exists, load it
   without any request. Otherwise GET play-by-play with timeout=30 and raise_for_status().
4. Do not catch exceptions: a failed request must raise and nothing must be saved.
5. Before saving, raise ValueError if the response id is different from game_id or if "plays" is not a list.
6. Use UTF-8 files, and time.sleep(1) after each download.
7. For each game build pandas.json_normalize(plays) and add a game_id column. Finally, RETURN pandas.concat of all games.

Return only the code in a single Python code block.
"""


generated_code = ask_qwen(model,tokenizer, code_prompt,system="You write concise, correct Python code.", max_new_tokens=1400)

display(Markdown(generated_code))



[transformers] Both `max_new_tokens` (=1400) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


```python
import os
import pandas as pd
import requests
import time
from datetime import datetime, timedelta
import json
from typing import Optional

HOCKEY_DATA_DIR = os.getenv('HOCKEY_DATA_DIR')

def get_hockey_data(year: int, path: Optional[str] = None) -> pd.DataFrame:
    if path is None:
        path = HOCKEY_DATA_DIR
    if not path:
        raise ValueError("Neither path nor HOCKEY_DATA_DIR environment variable is set")
    
    season = f"{year}-{year+1}"
    season_dir = os.path.join(path, season)
    os.makedirs(season_dir, exist_ok=True)
    
    # Start date: September 1 of year
    start_date = datetime(year, 9, 1)
    # End date: October 31 of year + 1
    end_date = datetime(year + 1, 10, 31)
    
    game_ids = set()
    
    current_date = start_date
    while current_date <= end_date:
        date_str = current_date.strftime('%Y-%m-%d')
        url = f"https://api-web.nhle.com/v1/schedule/{date_str}"
        try:
            response = requests.get(url, timeout=30)
            response.raise_for_status()
            data = response.json()
            game_week = data.get('gameWeek', [])
            for day in game_week:
                for game in day.get('games', []):
                    game_id = game['id']
                    game_type = game['gameType']
                    if game_type in [2, 3] and str(game_id).startswith(str(year)):
                        game_ids.add(game_id)
        except requests.exceptions.RequestException as e:
            raise e
        current_date += timedelta(days=1)
        time.sleep(1)
    
    all_plays_data = []
    
    for game_id in game_ids:
        file_path = os.path.join(season_dir, f"{game_id}.json")
        
        if os.path.exists(file_path):
            with open(file_path, 'r', encoding='utf-8') as f:
                data = json.load(f)
        else:
            url = f"https://api-web.nhle.com/v1/gamecenter/{game_id}/play-by-play"
            try:
                response = requests.get(url, timeout=30)
                response.raise_for_status()
                data = response.json()
            except requests.exceptions.RequestException as e:
                raise e
            
            # Validate response
            if data.get('id') != game_id or not isinstance(data.get('plays'), list):
                raise ValueError(f"Invalid response for game_id {game_id}: id mismatch or plays not a list")
            
            # Save to file
            with open(file_path, 'w', encoding='utf-8') as f:
                json.dump(data, f, ensure_ascii=False)
        
        # Normalize plays and add game_id
        plays_df = pd.json_normalize(data['plays'])
        plays_df['game_id'] = game_id
        all_plays_data.append(plays_df)
    
    if not all_plays_data:
        return pd.DataFrame()
    
    return pd.concat(all_plays_data, ignore_index=True)
```

Runnable qwen code with manual corerctions applied

In [ ]:
import os
import pandas as pd
import requests
import time
from datetime import datetime, timedelta
import json
from typing import Optional

# manual fix 1
#HOCKEY_DATA_DIR = os.getenv('HOCKEY_DATA_DIR')

def get_hockey_data(year: int, path: Optional[str] = None) -> pd.DataFrame:
    #if path is None:
    #   path = HOCKEY_DATA_DIR
     # manual fix 1
    if path is None:
        path = os.getenv('HOCKEY_DATA_DIR')
    if not path:
        raise ValueError("Neither path nor HOCKEY_DATA_DIR environment variable is set")

    #Manual fix 2
    #season = f"{year}-{year+1}"
    season = f"{year}-{str(year + 1)[-2:]}"
    season_dir = os.path.join(path, season)
    os.makedirs(season_dir, exist_ok=True)

    # Start date: September 1 of year
    start_date = datetime(year, 9, 1)
    # End date: October 31 of year + 1
    end_date = datetime(year + 1, 10, 31)

    game_ids = set()

    current_date = start_date
    while current_date <= end_date:
        date_str = current_date.strftime('%Y-%m-%d')
        url = f"https://api-web.nhle.com/v1/schedule/{date_str}"
        try:
            response = requests.get(url, timeout=30)
            response.raise_for_status()
            data = response.json()
            game_week = data.get('gameWeek', [])
            for day in game_week:
                for game in day.get('games', []):
                    game_id = game['id']
                    game_type = game['gameType']
                    if game_type in [2, 3] and str(game_id).startswith(str(year)):
                        game_ids.add(game_id)
        except requests.exceptions.RequestException as e:
            raise e
        # Manual fix 3: each schedule request returns 7 days (61 requests instead of 426)
        # current_date += timedelta(days=1)
        current_date += timedelta(days=7)

        # Manual fix 4: moved after each game download
        # time.sleep(1)


    # Manual fix 5: raise instead of silently returning an empty DataFrame
    if not game_ids:
        raise ValueError(f"No games found for season {season}")
    all_plays_data = []

    for game_id in game_ids:
        file_path = os.path.join(season_dir, f"{game_id}.json")

        if os.path.exists(file_path):
            with open(file_path, 'r', encoding='utf-8') as f:
                data = json.load(f)
        else:
            url = f"https://api-web.nhle.com/v1/gamecenter/{game_id}/play-by-play"
            try:
                response = requests.get(url, timeout=30)
                response.raise_for_status()
                data = response.json()
            except requests.exceptions.RequestException as e:
                raise e

            # Validate response
            if data.get('id') != game_id or not isinstance(data.get('plays'), list):
                raise ValueError(f"Invalid response for game_id {game_id}: id mismatch or plays not a list")

            # Save to file
            with open(file_path, 'w', encoding='utf-8') as f:
                json.dump(data, f, ensure_ascii=False)
            #Manual fix 4: sleep only after an actual download
            time.sleep(1)

        # Normalize plays and add game_id
        plays_df = pd.json_normalize(data['plays'])
        plays_df['game_id'] = game_id
        all_plays_data.append(plays_df)

    if not all_plays_data:
        return pd.DataFrame()

    return pd.concat(all_plays_data, ignore_index=True)

##Comparing Manual code vs Generated code

We compare the fetching, caching, and loading behaviour of your manual and LLM-generated code using the same tests: a known game, a cache hit without a new request, and an unsuccessful request. Verify both implementations against expected results.

In [28]:
from tempfile import TemporaryDirectory
from unittest.mock import Mock, patch,call

import importlib.util
import json

from contextlib import contextmanager
from pathlib import Path


import requests


#We choose a specific game from a specific season for our existing play
season = 2023
game_id = 2023021190                    #  provided example from readme
#game_date = date(2023, 6, 3)            # day the fake schedule lists the game on
play_url = f"https://api-web.nhle.com/v1/gamecenter/{game_id}/play-by-play"

#Expected result: the real API response for the game.This is downloaded only once
game_response = requests.get(play_url, timeout=30).json()

# We fetch the get_hockey_data from manual implementation.This was loaded earlier from github
def manual_function(path):
    spec = importlib.util.spec_from_file_location("question_1", path)
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module.get_hockey_data

REPO_DIR=Path(globals().get("REPO_DIR", "."))

# The 2 functions we test
implementations = {
    "Manual": manual_function(f"{REPO_DIR}/ift6758/data/question_1.py"),
    "LLM_with_fix": get_hockey_data,
}


# Mock  NHL API endpoint
def mock_api_endpoint(status_code=200):
   #Fake NHL API. Every schedule week is smplified and  lists a  game_id.
   #Every play-by-play request returns `game_response`otherwise  raise_for_status().

    schedule = Mock()
    schedule.json.return_value = {"gameWeek": [{"games": [{"id": game_id, "gameType": 3}]}]}
    play_by_play = Mock()
    play_by_play.json.return_value = game_response
    if status_code != 200:
        play_by_play.raise_for_status.side_effect = requests.HTTPError(f"{status_code} Clientside Error")
    get = Mock(side_effect=lambda url, *args, **kwargs: schedule if "/schedule/" in url else play_by_play)
    return get, play_by_play

# Return only the play-by-play requests sent to the mock
def pbp_calls(mock):
    calls= []
    for a in mock.call_args_list:
        if "/play-by-play" in a.args[0]:
            calls.append(a)
    return calls

@contextmanager
def nhl_api(get):
    # requests.get for the LLM code and Session.get for the manual code
    with patch("requests.get", get), patch("requests.Session.get", get), patch("time.sleep"):
        yield


# this function will directly verify the output of both functions for all 3 cases listed in the readme
def compare(get_hockey_data):
    with TemporaryDirectory() as temp_dir:
        folder = Path(temp_dir) / "raw"

        # TEST 1: Known game
        get, play_by_play = mock_api_endpoint()
        with nhl_api(get):
            downloaded = get_hockey_data(season, folder)

        # Fetching:
        assert pbp_calls(get) == [call(play_url, timeout=30)], f"play-by-play requests: {pbp_calls(get)}"

        # Caching:
        #save  a list of files
        saved = list(folder.rglob(f"{game_id}.json"))
        #bug fix:prevents index error
        assert len(saved) == 1, "no cache file written"


        assert json.loads(saved[0].read_text(encoding="utf-8")) == game_response, "cached JSON does not match"

        # Loading:
        assert downloaded["eventId"].tolist() == [pl["eventId"] for pl in game_response["plays"]], "plays does not match actual"
        assert (downloaded["game_id"] == game_id).all(), "wrong game_id column"

        print(f"  1. Known game: data and cache file match the reference JSON "
              f"({get.call_count - 1} schedule requests . cache folder: '{saved[0].parent.name}/')")


        # 2. Cache hit: theremust be no play-by-play request
        get, _ = mock_api_endpoint()
        with nhl_api(get):
            from_cache = get_hockey_data(season, folder)
        assert not pbp_calls(get), "Cache was ignored"
        assert from_cache.equals(downloaded), "cached result differs from the first call"
        print("  2. Cache hit: no new play-by-play request, same DataFrame")

    # 3. Unsuccessful request: 404 error only with no cache
    with TemporaryDirectory() as temp_dir:
        folder = Path(temp_dir) / "raw"
        get, _ = mock_api_endpoint(status_code=404)
        with nhl_api(get):
            try:
                get_hockey_data(season, folder)
            except requests.HTTPError:
                pass
            else:
                raise AssertionError("no error raised for a failed request")
        assert not list(folder.rglob("*.json")), "a file was cached for a failed request"
        print("  3. Failed request: HTTPError raised, nothing cached")



def run_test(name, fetch):
    print(name)
    try:
        compare(fetch)
        print("  All tests passed")
    except Exception as e:
        print(f"  FAILED: {e}")

for name, fetch in implementations.items():
    run_test(name, fetch)


Manual
Retrieving NHL data for the 2023-24 season.
Number of games to process: 1
Processed 1/1 games
  1. Known game: data and cache file match the reference JSON (61 schedule requests . cache folder: '2023-24/')
Retrieving NHL data for the 2023-24 season.
Number of games to process: 1
Processed 1/1 games
  2. Cache hit: no new play-by-play request, same DataFrame
Retrieving NHL data for the 2023-24 season.
Number of games to process: 1
  3. Failed request: HTTPError raised, nothing cached
  All tests passed
LLM_with_fix
  1. Known game: data and cache file match the reference JSON (61 schedule requests . cache folder: '2023-24/')
  2. Cache hit: no new play-by-play request, same DataFrame
  3. Failed request: HTTPError raised, nothing cached
  All tests passed
